<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/Dask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dask para una sesión de computación distribuida

<img src="https://docs.dask.org/en/latest/_images/dask_horizontal.svg" align="right" width="32%" alt="Logo de Dask">

Este cuaderno está diseñado para enseñar **Dask desde cero** con una narrativa pensada para estudiantes que todavía no dominan la computación distribuida.

La idea pedagógica es esta:

- En **Colab** usamos Dask para entender conceptos, la API y ejemplos pequeños.
- En tu **infraestructura Docker local** mostramos el valor real del clúster con `scheduler`, `workers`, dashboard y benchmark contra Pandas.
- El objetivo no es decir que Dask siempre gana, sino entender **cuándo sí tiene sentido**.

## Objetivos de aprendizaje

Al finalizar la sesión, la persona debería poder:

1. Explicar qué es Dask y cómo se relaciona con Pandas.
2. Entender primero un flujo secuencial en Pandas antes de paralelizarlo.
3. Entender la diferencia entre `client`, `scheduler`, `workers` y `particiones`.
4. Reconocer que Dask evalúa de forma perezosa (`lazy`).
5. Comparar un flujo simple en Pandas vs Dask.
6. Ejecutar un benchmark real sobre el clúster Docker del curso.

## Estrategia didáctica sugerida

1. Empezar **secuencialmente** con Python/Pandas.
2. Mostrar el cuello de botella conceptual: una máquina, un proceso, una memoria.
3. Repetir la misma lógica con Dask para que el estudiante vea qué cambia y qué no.
4. Explicar `delayed`, `compute()` y `persist()`.
5. Cerrar con el clúster real en Docker y el dashboard.

## Imagen guía: estructura de nodos y distribución

Esta imagen es útil para explicar la intuición del clúster antes de entrar al código.

<img src="https://raw.githubusercontent.com/jazaineam1/BigData2026/refs/heads/main/Images/Cluster/nodos.png" alt="Estructura de nodos" width="760"/>

In [1]:
# Si estás en Colab y hace falta instalar dependencias, descomenta esta celda y ejecútala una vez.
# %pip install -q "dask[distributed,dataframe]==2024.5.1" pandas pyarrow graphviz

In [2]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import dask
import dask.dataframe as dd
from dask.distributed import Client, wait
from IPython.display import display

try:
    from google.colab import files  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"Ejecutando en Colab: {IN_COLAB}")

c:\Users\nib1l\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


AttributeError: module 'pandas.core.strings' has no attribute 'StringMethods'

## 1. Idea mental: ¿qué problema resuelve Dask?

Pandas funciona muy bien cuando:

- los datos caben cómodamente en memoria,
- el trabajo cabe en una sola máquina,
- y el costo de coordinar un sistema distribuido sería mayor que el beneficio.

Dask entra en escena cuando queremos:

- dividir datos en **particiones**,
- procesarlas en paralelo,
- trabajar con un API muy parecida a Pandas,
- o conectarnos a un clúster con varios workers.

### Mapa mental simple

- **Client**: desde donde tú envías el trabajo.
- **Scheduler**: quien coordina qué tarea se ejecuta y en qué worker.
- **Worker**: proceso que hace cómputo y usa memoria.
- **Partition**: fragmento del dataset; en Dask DataFrame cada partición se parece a un DataFrame pequeño de Pandas.

## Imagen conceptual de Dask DataFrame

Esta imagen ayuda a explicar que un Dask DataFrame es una colección de múltiples particiones, no un único bloque monolítico en memoria.

<img src="https://docs.dask.org/en/stable/_images/dask-dataframe.svg" align="center" width="780" alt="Dask DataFrame">

## 2. Empezar secuencialmente: primero Pandas

Para clase, es mejor no saltar de inmediato a la computación distribuida.

Primero queremos que el estudiante vea un flujo normal:

- leer datos,
- agrupar,
- calcular una métrica,
- interpretar el resultado.

Después sí mostramos cómo Dask toma esa misma idea y la ejecuta por particiones.

In [ ]:
ventas_secuencial = pd.DataFrame(
    {
        "region": ["Norte", "Sur", "Norte", "Centro", "Sur", "Occidente"],
        "ventas": [120, 80, 200, 150, 90, 300],
        "unidades": [2, 1, 4, 3, 1, 5],
    }
)

ventas_por_region_pd = ventas_secuencial.groupby("region")[["ventas", "unidades"]].sum()

print("DataFrame original en Pandas")
display(ventas_secuencial)

print("Resultado secuencial con Pandas")
display(ventas_por_region_pd)

### Mensaje pedagógico

Aquí todavía no hay clúster, ni workers, ni scheduler.

Solo hay una idea sencilla: **tomar datos tabulares y aplicar transformaciones**.
Esa misma idea la vamos a conservar cuando pasemos a Dask.

## 3. La misma idea, ahora con Dask DataFrame

Ahora sí repetimos la misma lógica, pero con Dask. Así el estudiante compara:

- misma intención analítica,
- sintaxis parecida,
- ejecución distinta.

## 4. Primer contacto con `dask.delayed`

Después de comparar Pandas y Dask DataFrame, conviene mostrar el corazón de Dask: construimos un grafo de tareas y solo se ejecuta cuando llamamos `compute()`.

In [ ]:
@dask.delayed
def cuadrado(x):
    return x ** 2

@dask.delayed
def suma(valores):
    return sum(valores)

tareas = [cuadrado(i) for i in range(1, 6)]
resultado = suma(tareas)

print("Objeto perezoso:", resultado)
print("Resultado real:", resultado.compute())

**Mensaje clave para clase:** hasta que no llamamos `compute()`, Dask solo está construyendo el plan de ejecución.

## 5. Levantar un clúster local pequeño dentro del notebook

En Colab y en una sesión local sencilla podemos usar un clúster pequeño en la misma máquina solo para ilustrar conceptos.

Esto **no reemplaza** el clúster Docker del curso; solo nos ayuda a enseñar la API.

In [ ]:
try:
    client.close()
except Exception:
    pass

client = Client(
    processes=not IN_COLAB,
    n_workers=2,
    threads_per_worker=1,
    memory_limit="1GB",
)
client

In [ ]:
print("Dashboard:", client.dashboard_link)

## 3. La misma idea con Dask DataFrame: similar a Pandas, pero por particiones

In [ ]:
pdf_demo = pd.DataFrame(
    {
        "region": ["Norte", "Sur", "Norte", "Centro", "Sur", "Occidente"],
        "ventas": [120, 80, 200, 150, 90, 300],
        "unidades": [2, 1, 4, 3, 1, 5],
    }
)

ddf_demo = dd.from_pandas(pdf_demo, npartitions=3)

print("Tipo en Pandas:", type(pdf_demo))
print("Tipo en Dask:", type(ddf_demo))
print("Particiones:", ddf_demo.npartitions)

ddf_demo

In [ ]:
print("Primera partición como Pandas:")
display(ddf_demo.partitions[0].compute())

print("Agregación perezosa:")
agg_demo = ddf_demo.groupby("region")["ventas"].sum()
print(agg_demo)

print("Resultado materializado:")
display(agg_demo.compute())

### Punto pedagógico importante

Dask **no es magia**. Tiene costo de coordinación.

Por eso, con datasets pequeños, muchas veces:

- Pandas es más simple,
- Pandas es más rápido,
- y Dask solo añade overhead.

Eso es normal y es parte de lo que queremos enseñar.

### Punto pedagógico importante

La transición correcta para clase es:

1. primero Pandas secuencial,
2. luego Dask en pequeño,
3. y solo después distribución real.

Dask **no es magia**. Tiene costo de coordinación.

Por eso, con datasets pequeños, muchas veces:

- Pandas es más simple,
- Pandas es más rápido,
- y Dask solo añade overhead.

In [ ]:
## 6. Crear un dataset de ventas para comparar Pandas vs Dask

Aquí construiremos un dataset moderado, dividido en varios CSV. Esto nos permite simular el patrón típico de Dask: **muchos archivos, varias particiones y operaciones repetidas**.

In [ ]:
pdf = pd.concat((pd.read_csv(f) for f in archivos), ignore_index=True)
ddf = dd.read_csv(str(DATA_DIR / "ventas_*.csv"), blocksize=None)

print("Filas en Pandas:", f"{len(pdf):,}")
print("Particiones en Dask:", ddf.npartitions)

ddf.head()

## 6. Mismo pipeline de negocio en ambos frameworks

Vamos a construir columnas derivadas y luego dos agregaciones.

Esto es útil para clase porque separa muy bien dos ideas:

- la **lógica de análisis** es casi la misma,
- pero el **modelo de ejecución** cambia.

In [ ]:
## 7. Mismo pipeline de negocio en ambos frameworks

Vamos a construir columnas derivadas y luego dos agregaciones.

Esto es útil para clase porque separa muy bien dos ideas:

- la **lógica de análisis** es casi la misma,
- pero el **modelo de ejecución** cambia.

In [ ]:
# Benchmark con Pandas
inicio = perf_counter()
pdf_modelado = enriquecer_pandas(pdf)
resultado_pandas = resumen_ventas_pandas(pdf_modelado)
tiempo_pandas = perf_counter() - inicio

print(f"Tiempo Pandas: {tiempo_pandas:.2f} s")
display(resultado_pandas["ventas_por_region"])

In [ ]:
# Benchmark con Dask
inicio = perf_counter()
ddf_modelado = enriquecer_dask(ddf).persist()
wait(ddf_modelado)
resultado_dask_lazy = resumen_ventas_dask(ddf_modelado)
resultado_dask = dask.compute(*resultado_dask_lazy.values())
tiempo_dask = perf_counter() - inicio

resultado_dask = dict(zip(resultado_dask_lazy.keys(), resultado_dask, strict=True))
resultado_dask = {k: v.sort_index() if hasattr(v, "sort_index") else v for k, v in resultado_dask.items()}
print(f"Tiempo Dask: {tiempo_dask:.2f} s")
display(resultado_dask["ventas_por_region"])

In [ ]:
pd.testing.assert_frame_equal(
    resultado_pandas["ventas_por_region"],
    resultado_dask["ventas_por_region"],
    check_exact=False,
    rtol=1e-6,
    atol=1e-6,
)

pd.testing.assert_frame_equal(
    resultado_pandas["ventas_por_region_categoria"],
    resultado_dask["ventas_por_region_categoria"],
    check_exact=False,
    rtol=1e-6,
    atol=1e-6,
)

comparacion = pd.DataFrame(
    {
        "framework": ["pandas", "dask"],
        "tiempo_s": [tiempo_pandas, tiempo_dask],
        "observación": [
            "Sin overhead distribuido",
            "Incluye coordinación y ejecución por particiones",
        ],
    }
)

comparacion["speedup_vs_pandas"] = comparacion.loc[0, "tiempo_s"] / comparacion["tiempo_s"]
comparacion

### Cómo interpretar este resultado en clase

Si Dask sale más lento aquí, **no pasa nada**. De hecho, eso ayuda a enseñar una lección importante:

- Dask tiene overhead.
- En datos pequeños o medianos, Pandas suele ganar.
- Dask empieza a brillar cuando aumentan los datos, las particiones o la necesidad de usar varios workers.

La pregunta correcta no es Ã¢â‚¬Å“Ã‚Â¿Dask es mejor que Pandas?Ã¢â‚¬Â sino:

**¿este problema justifica paralelismo y/o distribución?**

### Cómo interpretar este resultado en clase

Si Dask sale más lento aquí, **no pasa nada**. De hecho, eso ayuda a enseñar una lección importante:

- Dask tiene overhead.
- En datos pequeños o medianos, Pandas suele ganar.
- Dask empieza a brillar cuando aumentan los datos, las particiones o la necesidad de usar varios workers.

La pregunta correcta no es Ã¢â‚¬Å“Ã‚Â¿Dask es mejor que Pandas?Ã¢â‚¬Â sino:

**¿este problema justifica paralelismo y/o distribución?**

In [ ]:
## 8. Visualizar el grafo de tareas

Esto ayuda mucho a explicar que Dask construye un DAG (grafo acíclico dirigido) antes de ejecutar.

## 8. Qué mirar en el dashboard de Dask

Mientras corre una tarea distribuida, abre el dashboard y observa:

- **Task Stream**: muestra si las tareas realmente se están repartiendo entre workers.
- **Workers**: cuántos workers hay, cuánta memoria usan y si alguno está saturado.
- **Graph**: deja ver la estructura del trabajo.
- **System / Memory**: útil para explicar que el cuello de botella no siempre es CPU.

Referencia oficial: la documentación de Dask describe el dashboard distribuido como una de las herramientas principales de diagnóstico.

## 9. Qué mirar en el dashboard de Dask

Mientras corre una tarea distribuida, abre el dashboard y observa:

- **Task Stream**: muestra si las tareas realmente se están repartiendo entre workers.
- **Workers**: cuántos workers hay, cuánta memoria usan y si alguno está saturado.
- **Graph**: deja ver la estructura del trabajo.
- **System / Memory**: útil para explicar que el cuello de botella no siempre es CPU.

Referencia oficial: la documentación de Dask describe el dashboard distribuido como una de las herramientas principales de diagnóstico.

## 9. Conectar este tema con tu clúster Docker del curso

Aquí está la parte importante para la sesión: **Colab no es el lugar ideal para demostrar cómputo distribuido real con tu Docker local**.

Entonces conviene separar los objetivos:

- **En Colab**: teoría, API, ejemplos pequeños y noción de particiones.
- **En local/Jupyter del curso**: conexión al scheduler real, varios workers y benchmark pesado.

Tu infraestructura relevante está en:

- `infraestructura/docker-compose.yml`
- `infraestructura/dask/jobs/benchmark_dask_vs_pandas.py`

El `docker-compose` monta `C:/Users/nib1l/Documents/diplocopia/BigData2026/infraestructura/dask/jobs:/app`, así que el benchmark puede ejecutarse directamente dentro del clúster.

## 10. Conectar este tema con tu clúster Docker del curso

Aquí está la parte importante para la sesión: **Colab no es el lugar ideal para demostrar cómputo distribuido real con tu Docker local**.

Entonces conviene separar los objetivos:

- **En Colab**: teoría, API, ejemplos pequeños y noción de particiones.
- **En local/Jupyter del curso**: conexión al scheduler real, varios workers y benchmark pesado.

Tu infraestructura relevante está en:

- `infraestructura/docker-compose.yml`
- `infraestructura/dask/jobs/benchmark_dask_vs_pandas.py`

El `docker-compose` monta `C:/Users/nib1l/Documents/diplocopia/BigData2026/infraestructura/dask/jobs:/app`, así que el benchmark puede ejecutarse directamente dentro del clúster.

In [ ]:
USE_DOCKER_CLUSTER = False
SCHEDULER_ADDRESS = "tcp://localhost:8786"  # usa tcp://dask-scheduler:8786 si corres dentro del Jupyter del docker-compose

docker_client = None
if USE_DOCKER_CLUSTER:
    docker_client = Client(SCHEDULER_ADDRESS)
    display(docker_client)
    docker_info = docker_client.scheduler_info()
    display(pd.DataFrame(
        [
            {
                "worker": direccion,
                "threads": meta.get("nthreads"),
                "memory_limit_mb": round(meta.get("memory_limit", 0) / 1024**2, 1),
            }
            for direccion, meta in docker_info.get("workers", {}).items()
        ]
    ))
else:
    print("Activa USE_DOCKER_CLUSTER=True cuando ejecutes este notebook en tu equipo o en el Jupyter del entorno Docker.")

## 10. Comandos recomendados para la demostracion real con Docker

Ejecuta estos comandos desde la carpeta `infraestructura/`. En Colab quedan como referencia; la demostracion real debe hacerse en tu maquina local o en el Jupyter del entorno Docker.

```bash
cd infraestructura

docker compose --profile dask up -d --scale dask-worker=4 dask-scheduler dask-worker

docker compose ps

docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py \
  --parts 6 \
  --rows-per-part 1000000 \
  --rebuild-data
```

Si quieres una prueba mas ligera para clase, empieza con:

```bash
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py \
  --parts 6 \
  --rows-per-part 250000 \
  --rebuild-data
```

## 11. Comandos recomendados para la demostracion real con Docker

Ejecuta estos comandos desde la carpeta `infraestructura/`. En Colab quedan como referencia; la demostracion real debe hacerse en tu maquina local o en el Jupyter del entorno Docker.

```bash
cd infraestructura

docker compose --profile dask up -d --scale dask-worker=4 dask-scheduler dask-worker

docker compose ps

docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py \
  --parts 6 \
  --rows-per-part 1000000 \
  --rebuild-data
```

Si quieres una prueba mas ligera para clase, empieza con:

```bash
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py \
  --parts 6 \
  --rows-per-part 250000 \
  --rebuild-data
```

## 11.1 Comandos reales en PowerShell para esta maquina

En este curso la ruta real fue:

```powershell
cd C:\Users\nib1l\Documents\diplocopia\BigData2026\infraestructura
docker compose --profile dask up -d dask-scheduler dask-worker --scale dask-worker=4
docker compose --profile dask ps
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 6 --rows-per-part 1000 --rebuild-data
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 6 --rows-per-part 250000 --rebuild-data
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 6 --rows-per-part 500000 --rebuild-data
docker compose exec dask-scheduler python /app/benchmark_dask_vs_pandas.py --parts 24 --rows-per-part 500000 --rebuild-data
docker compose logs --tail=50 dask-scheduler
docker compose logs --tail=50 dask-worker
```

## 11.2 Resultados observados en el benchmark real

Estos tiempos salieron al ejecutar `benchmark_dask_vs_pandas.py` dentro del contenedor `dask-scheduler`, con validacion correcta entre Pandas y Dask.

| Partes x filas | Filas totales | Pandas | Dask | Lectura |
|---|---:|---:|---:|---|
| 6 x 1,000 | 6,000 | 0.13 s | 8.49 s | Pandas gana por mucho; Dask paga overhead |
| 6 x 50,000 | 300,000 | 1.09 s | 2.90 s | Todavia gana Pandas |
| 6 x 250,000 | 1,500,000 | 6.07 s | 8.49 s | La brecha se reduce, pero Pandas sigue ganando |
| 6 x 500,000 | 3,000,000 | 15.50 s | 11.92 s | Primera corrida donde Dask supero a Pandas |
| 24 x 500,000 | 12,000,000 | 71.39 s | 72.20 s | Quedan practicamente empatados |

### Lectura pedagogica importante

En esta maquina **Pandas gana claramente en datos pequenos y medianos**. El **primer caso medido** donde Dask fue mas rapido aparecio en `6 x 500,000 = 3,000,000` filas. Sin embargo, ese cruce **no fue perfectamente estable** al repetir con distinta configuracion de workers, asi que la conclusion correcta no es "a partir de X filas Dask siempre gana", sino:

- Dask empieza a competir cuando el volumen crece.
- El resultado depende del costo de coordinacion, del `shuffle`, del numero de particiones y de la carga real del cluster.
- En un benchmark con mucho `set_index()` y `shuffle`, el costo distribuido puede comerse parte del beneficio.


## 11.3 Que hacen los workers y de que depende tener mas

Un **worker** es un proceso que ejecuta tareas, usa CPU, usa memoria y guarda particiones temporales. Tener mas workers **no garantiza** mejor tiempo por si solo.

El beneficio de aumentar workers depende de:

- **Particiones suficientes**: si solo tienes 6 particiones, no siempre aprovechas bien 8 workers.
- **CPU real disponible**: si Docker Desktop o el equipo no tienen suficientes nucleos libres, los workers compiten entre si.
- **Memoria**: mas workers reparten memoria, pero tambien aumentan la coordinacion y el movimiento de datos.
- **Tipo de trabajo**: operaciones con `shuffle` fuerte, como `set_index()`, suelen costar bastante en distribuido.
- **I/O y red**: si el cuello de botella es leer, escribir o mover datos, agregar workers puede no ayudar mucho.
- **Overhead del scheduler**: mas workers significa mas coordinacion.

En otras palabras: mas workers ayudan cuando hay suficiente trabajo paralelo, suficientes particiones y recursos reales para sostenerlos.


## 11.4 Prueba real cambiando el numero de workers

Se repitio el mismo benchmark de `6 x 500,000 = 3,000,000` filas cambiando el numero de workers.

| Workers pedidos | Workers detectados por Dask | Pandas | Dask | Lectura |
|---:|---:|---:|---:|---|
| 1 | 1 | 9.33 s | 18.30 s | Con un solo worker Dask pierde casi todo el beneficio distribuido |
| 2 | 2 | 7.58 s | 11.34 s | Mejora frente a 1 worker, pero Pandas sigue ganando |
| 4 | 4 | 7.35 s | 11.18 s | En esta corrida 4 workers no bastaron para superar a Pandas |
| 8 | 5 detectados | 8.72 s | 12.92 s | Pedir mas contenedores no implico mas capacidad efectiva en este entorno |

### Que ensena esta prueba

- Mas workers no siempre reducen el tiempo linealmente.
- Si el dataset tiene pocas particiones respecto al numero de workers, parte del cluster queda subutilizado.
- En esta maquina, al pedir 8 workers, el scheduler solo vio 5 workers activos durante la prueba. Eso muestra que el limite real tambien depende del entorno Docker, de la memoria disponible y del arranque efectivo de los procesos.
- La pregunta correcta no es "cuantos workers mas puedo crear?", sino "mi carga tiene suficiente paralelismo y recursos para aprovecharlos?".


## 11.5 Si solo tienes un PC, cuando conviene Dask y cuando conviene Pandas

Con un solo PC, **Pandas** suele ser la mejor opcion cuando los datos caben bien en RAM y el analisis termina rapido. Tiene menos overhead, es mas simple de depurar y evita el costo de coordinacion entre procesos.

**Dask** empieza a tener sentido en un solo PC cuando los datos ya vienen en muchos archivos, cuando quieres procesar por particiones, cuando Pandas empieza a presionar memoria o cuando quieres preparar el pipeline para escalar despues a varios workers o a otra infraestructura.

La idea clave no es "Dask siempre es mas rapido", sino "Dask escala mejor cuando el problema deja de ser comodo para Pandas".


### Que debe ver el estudiante en esa ejecucion real

1. Pandas usa un solo proceso y concentra memoria en una sola maquina.
2. Dask detecta varios workers y reparte tareas.
3. En el dashboard aparecen tareas simultaneas en el **Task Stream**.
4. El benchmark valida que el resultado analitico sea equivalente en ambos casos.
5. Si el dataset crece, Dask puede empezar a superar a Pandas o, al menos, volverse mas sostenible en memoria.


## 12. ¿Cuándo usar Pandas y cuándo usar Dask?

| Situación | Mejor opción | Razón |
|---|---|---|
| Dataset pequeño y análisis rápido | Pandas | Menos overhead y depuración más simple |
| Datos medianos repartidos en muchos archivos | Dask | Lee y procesa por particiones |
| Necesitas paralelizar en varios workers | Dask | Aprovecha scheduler + workers |
| Todo cabe fácil en RAM y el flujo es simple | Pandas | Más directo |
| Quieres mostrar arquitectura de cómputo distribuido | Dask | Hace visible la coordinación entre nodos |
| ETL local grande antes de Spark | Dask | Buen puente entre Pandas y sistemas distribuidos mayores |

## 13. Preguntas para discusión en clase

1. ¿Por qué Dask puede ser más lento que Pandas en un ejemplo pequeño?
2. ¿Qué representa una partición en Dask DataFrame?
3. ¿Qué hace el scheduler y qué hacen los workers?
4. ¿Por qué `compute()` cambia tanto el comportamiento del programa?
5. ¿Qué evidencia del dashboard demuestra paralelismo real?
6. ¿Qué pasaría si aumentamos el número de filas pero dejamos solo un worker?

## 14. Ejercicios propuestos

1. Cambia el número de particiones y observa si mejora o empeora el tiempo.
2. Agrega una nueva métrica por `category` y compara tiempos nuevamente.
3. Ejecuta el benchmark con `2` y luego con `4` workers.
4. Toma una captura del dashboard y explica qué está ocurriendo.
5. Repite el experimento con un dataset más pequeño y redacta por qué Pandas gana allí.

In [ ]:
## 15. Ideas de cierre para la sesión

- Dask no reemplaza a Pandas: lo extiende.
- El valor de Dask aparece cuando hay **particiones**, **concurrencia** y **tamaño** suficientes.
- En Colab lo usamos para comprender la idea.
- En el clúster Docker del curso lo usamos para **ver distribución real**.

## Referencias oficiales recomendadas

- Dask DataFrame Best Practices: https://docs.dask.org/en/stable/dataframe-best-practices.html
- Dask Distributed Diagnostics: https://docs.dask.org/en/stable/diagnostics-distributed.html
- Dask Delayed: https://docs.dask.org/en/stable/delayed.html